In [1]:

from mlflow.tracking import MlflowClient
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

The MlflowClient object allows us to interact with...

-- an MLflow Tracking Server that creates and manages experiments and runs.
-- an MLflow Registry Server that creates and manages registered models and model versions.

In [2]:
# we need to pass a tracking URI and/or a registry URI
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
experiments = client.search_experiments() # this will give a list of all the experiments
for exp in experiments:
    print(f"Experiment ID: {exp.experiment_id}, Name: {exp.name}, Artifact Location: {exp.artifact_location}")

Experiment ID: 1, Name: nyc-taxi-experiment, Artifact Location: /workspaces/MLOps-Model-Development-to-Production-deployment/Experiment Tracking using MLFlow/mlruns/1
Experiment ID: 0, Name: Default, Artifact Location: mlflow-artifacts:/0


In [3]:
experiments

[<Experiment: artifact_location=('/workspaces/MLOps-Model-Development-to-Production-deployment/Experiment '
  'Tracking using MLFlow/mlruns/1'), creation_time=1749424860439, experiment_id='1', last_update_time=1749424860439, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1749423036188, experiment_id='0', last_update_time=1749423036188, lifecycle_stage='active', name='Default', tags={}>]

#### We can use the MlflowClient instance to:
- Create and manage experiments
- Create and manage runs
- Register a new version for the experiment with model_name = "nyx-taxi-model"
- Transition to different stages like "staging", "production", or "archived"

In [4]:
model_name = "nyx-taxi-model"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: None
version: 2, stage: Staging


/tmp/ipykernel_2122/3851817081.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [5]:
## Since the latest version is not dedicated to any stage, we stage it 
client.transition_model_version_stage(
    name=model_name,
    version=latest_versions[0].version,
    stage="Staging",  # or "Production" if you want to promote it to production
    archive_existing_versions=False  # this will archive the existing versions in the stage
)

/tmp/ipykernel_2122/434687244.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1750108397734, current_stage='Staging', description='', last_updated_timestamp=1750195643690, name='nyx-taxi-model', run_id='fd47eeb5d4024ee5ac6c52461f610b93', run_link='', source=('/workspaces/MLOps-Model-Development-to-Production-deployment/Experiment '
 'Tracking using '
 'MLFlow/mlruns/1/fd47eeb5d4024ee5ac6c52461f610b93/artifacts/models_mlflow'), status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=1>

In [6]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=2,
    description=f"The model version 2 was transitioned to Staging on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1750108407670, current_stage='Staging', description='The model version 2 was transitioned to Staging on 2025-06-17', last_updated_timestamp=1750195653134, name='nyx-taxi-model', run_id='70b80d8b30994722b6328979a8a6ef6e', run_link='', source=('/workspaces/MLOps-Model-Development-to-Production-deployment/Experiment '
 'Tracking using '
 'MLFlow/mlruns/1/70b80d8b30994722b6328979a8a6ef6e/artifacts/models_mlflow'), status='READY', status_message=None, tags={}, user_id=None, version=2>